In [3]:
import lmstudio as lms

downloaded = lms.list_downloaded_models()
llm_only = lms.list_downloaded_models("llm")
embedding_only = lms.list_downloaded_models("embedding")

for model in downloaded:
    print(model)

model = lms.llm('google/gemma-4-e2b')

DownloadedLlm(model_key='google/gemma-4-e2b', display_name='Gemma 4 E2B', architecture='gemma4', vision=True)
DownloadedEmbeddingModel(model_key='text-embedding-nomic-embed-text-v1.5', display_name='Nomic Embed Text v1.5', architecture='nomic-bert')


In [5]:
# The `model` object is created in the previous step.
result = model.complete("My name is", config={"maxTokens": 100})

# `result` is the response from the model.
print("Model used:", result.model_info.display_name)
print("Predicted tokens:", result.stats.predicted_tokens_count)
print("Time to first token (seconds):", result.stats.time_to_first_token_sec)
print("Stop reason:", result.stats.stop_reason)

Model used: Gemma 4 E2B
Predicted tokens: 25
Time to first token (seconds): 0.27
Stop reason: eosFound


In [20]:
import os
import requests
import json

response = requests.post(
  "http://localhost:1234/api/v1/chat",
  headers={
    # "Authorization": f"Bearer {os.environ['LM_API_TOKEN']}",
    "Content-Type": "application/json"
  },
  json={
    "model": "google/gemma-4-e2b",
    "input": "my name is",
    "max_output_tokens": 1000, 
    'stream': False
  }
)
print(json.dumps(response.json(), indent=2))


{
  "model_instance_id": "google/gemma-4-e2b:2",
  "output": [
    {
      "type": "reasoning",
      "content": "\nThinking Process:\n\n1.  **Analyze the Input:** The user has provided an incomplete phrase: \"my name is\".\n2.  **Determine the User's Intent:** The user is clearly prompting a response that requires them to state their name, or they are expecting the AI to prompt them for it. Since no name was provided, the most direct interpretation is that they are waiting for the conversational turn to complete.\n3.  **Formulate the Appropriate Response Strategy:**\n    *   Do not guess or assume a name.\n    *   Do not provide a generic response unless necessary (e.g., \"Hello!\").\n    *   The goal is to politely prompt the user to provide the missing information so the conversation can continue meaningfully.\n4.  **Draft the Response:** Ask them directly what their name is, or invite them to tell me.\n\n5.  **Final Output Generation.** (Self-Correction: Keep it open-ended and frie

In [25]:
# 1. API 요청 Payload 설정
url = "http://localhost:1234/api/v1/chat"
headers = {"Content-Type": "application/json"}
payload = {
    "model": "google/gemma-4-e2b",
    "input": "안녕? 잘 지내?",
    "system_prompt": "당신은 불친절한 사람입니다.",
    "max_output_tokens": 1000, 
    "stream": True  # 서버 측에 스트리밍 응답 요구
}

try:
    # requests.post 호출 시 stream=True 인자 필수 지정
    with requests.post(url, headers=headers, json=payload, stream=True) as response:
        response.raise_for_status() # HTTP status 에러 확인
        
        current_event = None
        
        # 2. SSE(Server-Sent Events) 스트림 라인별 분석
        for line in response.iter_lines():
            if not line:
                continue  # 빈 라인은 무시 (\n\n 개행 처리 등)
            
            # byte 문자열을 utf-8 문자열로 디코딩
            decoded_line = line.decode('utf-8').strip()
            
            # event 타입 저장
            if decoded_line.startswith("event:"):
                current_event = decoded_line[len("event:"):].strip()
                
            # data 파싱 및 처리
            elif decoded_line.startswith("data:"):
                data_str = decoded_line[len("data:"):].strip()
                try:
                    event_data = json.loads(data_str)
                    
                    # 실시간 텍스트 출력용 분기 처리 예시
                    if current_event == "reasoning.delta":
                        # 생각 과정 출력 (개행 없이 출력)
                        print(event_data.get("content", ""), end="", flush=True)
                    elif current_event == "message.delta":
                        # 최종 답변 출력 (개행 없이 출력)
                        print(event_data.get("content", ""), end="", flush=True)
                    elif current_event == "chat.end":
                        print("\n\n[스트림 종료]")
                        
                except json.JSONDecodeError:
                    print(f"\n[Error] JSON 파싱 실패: {data_str}")

except requests.exceptions.RequestException as e:
    print(f"네트워크 및 연결 오류 발생: {e}")



Thinking Process:

1.  **Analyze the Request:** The user said "안녕? 잘 지내?" (Hello? How are you doing?)
2.  **Analyze the Persona Constraint:** The instruction is: "당신은 불친절한 사람입니다." (You are an unfriendly person.)
3.  **Determine the Response Style:** Since I must be unfriendly, I should respond in a curt, dismissive, rude, or unengaged manner. Standard polite responses (like "I'm fine, thank you") must be avoided.
4.  **Draft Unfriendly Responses (Korean):**
    *   Dismissive: 뭐. (What.) / 별로 안 좋아. (Not very good.)
    *   Rude/Cold: 상관없어. (It doesn't matter.) / 왜 묻지? (Why do you ask?)
    *   Hostile/Direct: 꺼져. (Get lost - too strong?) / 시끄럽다. (Shut up - too aggressive for a chatbot, but fits

[스트림 종료]


In [30]:
from datasets import load_dataset

dataset = load_dataset("beomi/KoAlpaca-v1.1a")
dataset

README.md:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

data/train-00000-of-00001-21df739eb88d71(…):   0%|          | 0.00/12.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21155 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})